In [1]:
from collections import defaultdict
import gymnasium as gym
import numpy as np

def sample_policy(state):
    player_sum, dealer_card, usable_ace = state
    return 1 if player_sum < 20 else 0

def monte_carlo_prediction(env, policy, num_episodes, gamma=1.0):
    returns_sum = defaultdict(float)
    returns_count = defaultdict(float)

    V = defaultdict(float)

    for episode_num in range(1, num_episodes + 1):
        if episode_num % 10000 == 0:
            print(f"Simulating Episode {episode_num}/{num_episodes}...")

        episode = []
        state, info = env.reset()
        done = False

        while not done:
            action = policy(state)
            next_state, reward, terminated, truncated, info = env.step(action)
            done = terminated or truncated
            episode.append((state, action, reward))
            state = next_state

        G = 0.0
        visited_states = set()

        for i in range(len(episode) - 1, -1, -1):
            s, a, r = episode[i]
            G = gamma * G + r

            if s not in visited_states:
                visited_states.add(s)
                returns_sum[s] += G
                returns_count[s] += 1
                V[s] = returns_sum[s] / returns_count[s]

    return V

env = gym.make('Blackjack-v1', sab=True)

num_episodes = 50000
state_values = monte_carlo_prediction(env, sample_policy, num_episodes)

env.close()

print("\n--- Example State Value Estimates ---")
print("Format: (Player Sum, Dealer Card, Usable Ace) -> Expected Value")

test_states = [
    (13, 2, False),
    (19, 5, False),
    (21, 7, True),
    (15, 10, True)
]

for s in test_states:
    print(f"State {s}: Value = {state_values[s]:.4f}")


Simulating Episode 10000/50000...
Simulating Episode 20000/50000...
Simulating Episode 30000/50000...
Simulating Episode 40000/50000...
Simulating Episode 50000/50000...

--- Example State Value Estimates ---
Format: (Player Sum, Dealer Card, Usable Ace) -> Expected Value
State (13, 2, False): Value = -0.5354
State (19, 5, False): Value = -0.6930
State (21, 7, True): Value = 0.9958
State (15, 10, True): Value = -0.4338
